In [3]:
!pip3 install pygraphviz

     |████████████████████████████████| 118 kB 482 kB/s eta 0:00:01
  Created wheel for pygraphviz: filename=pygraphviz-1.7-cp38-cp38-linux_x86_64.whl size=183687 sha256=a2a456f25e95aa5e073d94e808174c5b24ea59229da37fa9a1243ebac599ef6f
  Stored in directory: /home/david/.cache/pip/wheels/77/c6/8b/566e1c26d0f9d41093e28d8e1768f1412d54846e3296114feb
Successfully built pygraphviz


In [9]:
!pip3 install networkx

     |████████████████████████████████| 1.6 MB 2.9 MB/s eta 0:00:01
  Attempting uninstall: decorator
    Found existing installation: decorator 5.0.9
    Uninstalling decorator-5.0.9:
      Successfully uninstalled decorator-5.0.9


In [104]:
 !pip3 install Verilog_VCD

  Created wheel for Verilog-VCD: filename=Verilog_VCD-1.11-py3-none-any.whl size=6869 sha256=2fe75df5fd240002bfa5dff60fbd624e369ea5d40bbb64c021398dc9339c2e24
  Stored in directory: /home/david/.cache/pip/wheels/b6/3e/9e/26678a34ff6dbbacaefc45486fe00fde69c2c71cb5da6bd11c
Successfully built Verilog-VCD


In [1]:
import Verilog_VCD.Verilog_VCD as vvcd

In [2]:
import pygraphviz as pgv

In [3]:
def snk_signal_name(e):
    snk_port = e.attr.get("headport")
    src, snk = e
    name = snk
    if snk_port:
        name = name + "_inst_" + snk_port
    return name

In [38]:
def to_timeseries(l, last_time):
    ts = []
    previous = l[0]
    for current in l:
        ts.extend([previous[1]]*(current[0]-previous[0]))
        previous = current
    ts.extend([previous[1]]*((last_time+1)-previous[0]))
    return ts

def create_net_dict(path):
    data = vvcd.parse_vcd(path)
    net_dict = {}
    last_time = max(r['tv'][-1][0] for r in data.values())
    for record in data.values():
        ts = to_timeseries(record["tv"], last_time)
        for net in record["nets"]:
            net_dict[tuple(net["hier"].split("."))+(net["name"],)] = ts
    return net_dict, last_time

In [94]:
def generate_steps(file_name):
    G = pgv.AGraph(file_name+".dot")
    G.layout(prog="dot")
    net_dict, last_time = create_net_dict(file_name+".vcd")
    def valid(e, ts):
        hier = ('TOP', 'test_lambda_mod', 'sr', snk_signal_name(e)+"_valid")
        return net_dict[hier][ts] == "1"

    def ready(e, ts):
        hier = ('TOP', 'test_lambda_mod', 'sr', snk_signal_name(e)+"_ready")
        return net_dict[hier][ts] == "1"
    timestep = 2 # start after reset
    changed = True
    while timestep<min(last_time, 200):
        print(timestep)
        changed = False
        for e in G.edges():
            prev = e.attr["color"], e.attr["style"]
            e.attr["color"] = "black"
            e.attr["style"] = "solid"
            e.attr["penwidth"] = 2
            if not valid(e, timestep):
                e.attr["style"] = "dashed"
            if ready(e, timestep):
                e.attr["color"] = "green"
            new = e.attr["color"], e.attr["style"]
            if new != prev:
                changed = True
                # make changed edges thick
                e.attr["penwidth"] = 4
        if changed:
            G.draw(f"svgs/{file_name}_step{timestep}.svg", format="svg", prog="dot")
            G.write(f"svgs/{file_name}_step{timestep}.dot")
        timestep += 2

In [95]:
generate_steps("test6")

2
4
6
8
10
12
14
16
18
20
22
24
26
28
30
32
34
36
38
40
42
44
46
48
50
52
54
56
58
60
62
64
66
68
70
72
74
76
78
80
82
84
86
88
90
92
94
96
98
100
102
104
106
108
110
112
114
116
118
120
122
124
126
128
130
132
134
136
138
140
142
144
146
148
150
152
154
156
158
160
162
164
166
168
170
172
174
176
178
180
182
184
186
188
190
192
194
196
198


In [11]:
import networkx as nx

In [33]:
G = pgv.AGraph("test6.dot")

In [34]:
# remove back edges
for edge in G.edges():
    if edge.attr.get("dir") == "back":
        attr = dict(edge.attr.items())
        attr.pop("dir")
        headport = attr.pop("headport")
        tailport = attr.pop("tailport")
        attr["headport"] = tailport
        attr["tailport"] = headport
        src, snk = edge
        G.remove_edge(edge)
        G.add_edge(snk,src,**attr)

In [35]:
G.write("test6_no_back.dot")

In [36]:
nG = nx.nx_agraph.from_agraph(G)

In [44]:
net_dict, last_time = create_net_dict("test6.vcd")

In [ ]:
timestep = 60

In [45]:
def get_node_state(node, port, rv):
    name = node
    if port:
        name = name + "_inst_" + snk_port
    hier = ('TOP', 'test_lambda_mod', 'sr', name+"_"+rv)
    return net_dict[hier][timestep] == "1"

In [87]:
def get_in_ports(node):
    result = []
    sink = node
    for src in nG.predecessors(node):
        result.append(nG[src][sink][0].get("headport"))
    return result

def get_out_ports(node):
    result = []
    src = node
    for sink in nG.successors(node):
        result.append(nG[src][sink][0].get("tailport"))
    return result

def get_in_src(node, port):
    sink = node
    for src in nG.predecessors(node):
        if nG[src][sink][0]["headport"] == port:
            src_port = nG[src][sink][0].get("tailport")
            return src, src_port

def get_out_sink(node, port):
    src = node
    for sink in nG.successors(node):
        if nG[src][sink][0]["tailport"] == port:
            sink_port = nG[src][sink][0].get("headport")
            return sink, sink_port

In [90]:
def find_cycle(node, port, direction="ready", path=None):
    if path is None:
        path = []
    if port is None:
        return # ignore arguments and results
    new_path = path+[(node, port)]
    
    if (node, port) in path:
        print("cycle: ", path)
    elif direction == "ready" and "o" in port:
        sink, sink_port = get_out_sink(node, port)
        find_cycle(sink, sink_port, "ready", new_path)
    elif direction == "valid" and "i" in port:
        src, src_port = get_in_src(node, port)
        find_cycle(src, src_port, "valid", new_path)
    elif direction == "ready" and "i" in port:
        # in ports
        if "HLS_MERGE" in node:
            pass # in ready only depends on output
        elif "HLS_NDMUX" in node and port != "i0":
            # pred in needs to be valid
            find_cycle(node, "i0", "valid", new_path)
        else:
            # in ready depends on all ins being valid
            for i_port in get_in_ports(node): #TODO: all other?
                find_cycle(node, i_port, "valid", new_path)
        # out ports
        if "HLS_BUF" in node:#TODO: pred buf?
            pass # in and out are decoupled
        else:
            for o_port in get_out_ports(node):
                find_cycle(node, o_port, "ready", new_path)
    elif direction == "valid" and "o" in port:
        # in ports
        if "HLS_BUF" in node:#TODO: pred buf?
            pass # in and out are decoupled
        else:
            for i_port in get_in_ports(node): #TODO: all other?
                find_cycle(node, i_port, "valid", new_path)
        # out ports
        if "HLS_FORK" in node:
            for o_port in get_out_ports(node):
                find_cycle(node, o_port, "ready", new_path)

In [91]:
find_cycle('node_HLS_BRANCH_26', "o0")

In [56]:
[_ for _ in nG.nodes() if "BRANCH_26" in _]

['node_HLS_BRANCH_26']

In [57]:
nG.edges('node_HLS_BRANCH_26', data=True)

OutMultiEdgeDataView([('node_HLS_BRANCH_26', 'node_HLS_NDMUX_25', {'arrowhead': 'normal', 'color': 'black', 'fontsize': '10', 'headport': 'i2', 'labelangle': '45', 'labeldistance': '2.0', 'labelfontcolor': 'black', 'tailport': 'o1'}), ('node_HLS_BRANCH_26', 'node_HLS_MERGE_11', {'arrowhead': 'normal', 'color': 'black', 'fontsize': '10', 'headport': 'i1', 'labelangle': '45', 'labeldistance': '2.0', 'labelfontcolor': 'black', 'tailport': 'o0'})])

In [ ]:
nG.successors